# Benchmark simples de tags MOPEP com Ollama

Fluxo: carregar e tratar os dados, conectar ao Ollama, testar um caso em dois modelos, executar um lote de N exemplos e calcular F1 individual e macro.

## 1. Imports e configurações

In [ ]:
import ast
import os
from pathlib import Path
from time import perf_counter

import pandas as pd
from ollama import Client
from sklearn.metrics import f1_score
from sklearn.preprocessing import MultiLabelBinarizer
from tqdm.auto import tqdm

In [ ]:
DATASET_PATH = Path(
    os.getenv("MOPEP_BMC_DATA_DIR", "/tmp/mopep-bmc-tags")
) / "calibration.csv"

OLLAMA_HOST = os.getenv("OLLAMA_HOST", "http://127.0.0.1:8000")
MODELS = [
    os.getenv("OLLAMA_MODEL_1", "qwen2.5:7b-instruct-q4_K_M"),
    os.getenv("OLLAMA_MODEL_2", "qwen2.5:3b-instruct-q4_K_M"),
]

CASE_ID = os.getenv(
    "MOPEP_EXAMPLE_ID",
    "280847b6a4a911999bd1d2dde3ac0106a1fd552f8cf26dd00ca765f9834c155f",
)
N_REQUESTS_PER_MODEL = int(os.getenv("N_REQUESTS_PER_MODEL", "10"))
RANDOM_STATE = 42
OLLAMA_OPTIONS = {"temperature": 0.0, "num_predict": 512, "seed": 42}

## 2. Carregar e tratar os dados

O único tratamento necessário aqui é transformar a coluna textual `tags` em uma lista normalizada e remover linhas sem entrada ou rótulo.

In [ ]:
def parse_tags(value):
    try:
        tags = value if isinstance(value, list) else ast.literal_eval(str(value))
    except (SyntaxError, ValueError):
        return []

    if not isinstance(tags, list):
        return []

    return sorted({
        " ".join(str(tag).strip().casefold().split())
        for tag in tags
        if str(tag).strip()
    })


raw_df = pd.read_csv(DATASET_PATH)
raw_df["expected_tags"] = raw_df["tags"].apply(parse_tags)

dataset_df = (
    raw_df.loc[
        raw_df["example_id"].notna()
        & raw_df["business_model"].notna()
        & raw_df["expected_tags"].str.len().gt(0)
    ]
    .drop_duplicates("example_id")
    .reset_index(drop=True)
)

print(f"{len(dataset_df)} de {len(raw_df)} linhas prontas para avaliação")
dataset_df[["example_id", "domain", "expected_tags"]].head()

In [ ]:
dataset_df.loc[0]

## 3. Prompt de classificação

O prompt permanece explícito para que o experimento seja reproduzível.

In [ ]:
PROMPT_TAGS = """
Seu objetivo é analisar o modelo de negócios de uma empresa e classificá-la usando somente as tags da lista abaixo:

- empresa estabelecida: As evidências indicam que a empresa já é estabelecida, com mais de 5 anos de existência.
- estabelecimento físico: As evidências apontam que a empresa possui um estabelecimento físico ou escritório onde a atividade principal é desempenhada.
- b2b: A empresa vende produtos ou serviços para outras empresas.
- vendas remotas: As negociações e vendas são realizadas remotamente, por telefone, e-mail, WhatsApp, videoconferência ou outros canais digitais, sem necessidade de visita presencial.
- serviço: A empresa oferece soluções intangíveis, como consultoria, manutenção, suporte técnico ou treinamentos, entregando valor por meio de conhecimento e experiência.
- atuação nacional: A empresa realiza operações ou atende clientes em todo o território brasileiro, com presença ou cobertura em múltiplas regiões ou estados.
- tecnologia: A empresa se apresenta como inovadora no uso de tecnologia em sua atuação, produtos ou serviços.
- produto: A empresa comercializa bens tangíveis ou digitais, como equipamentos, roupas, alimentos, softwares, e-books ou outros produtos.
- whatsapp: O site possui indícios de que a empresa realiza atendimento ou vendas por WhatsApp.
- b2c: A empresa vende produtos ou serviços diretamente para o consumidor final.
- instagram: O site possui link apontando para o perfil da empresa no Instagram.
- compra planejada: Os clientes costumam pesquisar, comparar alternativas ou planejar antes de comprar os produtos ou contratar os serviços.
- blog: O site possui um blog ou área dedicada à publicação de artigos, notícias ou conteúdos informativos.
- público: empresas médias e grandes: A empresa B2B ou B2B2C se posiciona com ênfase no atendimento a empresas médias e grandes, com maior faturamento, estrutura ou número de funcionários.
- linkedin: O site possui link apontando para o perfil da empresa no LinkedIn.
- atuação regional: A empresa atua predominantemente em uma região geográfica específica, como um estado, grupo de municípios ou área delimitada.
- facebook: O site possui link apontando para o perfil da empresa no Facebook.
- atuação internacional: A empresa possui operações, clientes ou negócios em mais de um país, atuando em mercados estrangeiros ou globalmente.
- público: micro e pequenas empresas: A empresa B2B ou B2B2C se posiciona com ênfase no atendimento a MEIs, microempresas ou pequenas empresas.
- segmento: tecnologia da informação: A empresa tem como atividade principal a oferta de soluções de tecnologia da informação, como desenvolvimento de software, infraestrutura, redes, segurança da informação ou suporte técnico.

O modelo de negócios da empresa é:

{business_model}

Retorne somente uma lista Python com as tags aplicáveis.

Exemplo de resposta:

['empresa estabelecida', 'b2b', 'serviço', 'vendas remotas']

Procedimento:
- Avalie individualmente cada uma das 20 tags antes de responder.
- Considere todas as informações presentes no texto, mesmo quando algumas seções do Business Model Canvas estiverem vazias.
- Inclua uma tag quando houver evidência direta ou uma inferência razoável sustentada pelo texto.
- Não deixe de classificar a empresa apenas porque o documento está incompleto.
- Use [] somente depois de avaliar todas as tags e concluir que nenhuma delas pode ser sustentada.
- Retorne somente a lista final, sem mostrar a análise.

Regras:
- Use somente as tags apresentadas na lista.
- Copie exatamente o nome das tags, incluindo acentos.
- Não crie tags novas.
- Não inclua explicações ou qualquer texto fora da lista.
# - Não inclua uma tag quando não houver evidência suficiente.
# - Caso nenhuma tag seja aplicável, retorne [].
""".strip()

## 4. Conectar ao Ollama

O servidor Ollama e o túnel `uff-delta` devem estar ativos antes desta célula. Aqui iniciamos somente o cliente, validamos a conexão e mostramos os modelos disponíveis.

In [ ]:
client = Client(host=OLLAMA_HOST)
available_models = [model.model for model in client.list().models]

print(f"Conexão OK: {OLLAMA_HOST}")
print("Modelos disponíveis:")
for model in available_models:
    print(f"- {model}")

## 5. Funções pequenas: inferir e comparar

In [ ]:
def parse_model_output(raw_response):
    text = raw_response.strip()
    if text.startswith("```"):
        text = text.split("\n", 1)[-1].removesuffix("```").strip()

    try:
        parsed = ast.literal_eval(text)
    except (SyntaxError, ValueError):
        return [], False

    if not isinstance(parsed, list):
        return [], False

    return parse_tags(parsed), True

In [ ]:
def predict_one(example_id, model):
    rows = dataset_df.loc[dataset_df["example_id"].eq(example_id)]
    if rows.empty:
        raise ValueError(f"example_id não encontrado: {example_id}")

    example = rows.iloc[0]
    prompt = PROMPT_TAGS.format(business_model=example["business_model"])

    started_at = perf_counter()
    response = client.chat(
        model=model,
        messages=[{"role": "user", "content": prompt}],
        think=False,
        options=OLLAMA_OPTIONS,
    )
    latency_ms = (perf_counter() - started_at) * 1000

    raw_response = response.message.content or ""
    generated_tags, parse_ok = parse_model_output(raw_response)

    return {
        "example_id": example_id,
        "model": model,
        "expected_tags": example["expected_tags"],
        "generated_tags": generated_tags,
        "raw_response": raw_response,
        "parse_ok": parse_ok,
        "latency_ms": latency_ms,
    }

In [ ]:
def compare_tags(expected_tags, generated_tags):
    expected = set(expected_tags)
    generated = set(generated_tags)

    correct = expected & generated
    precision = len(correct) / len(generated) if generated else 0.0
    recall = len(correct) / len(expected) if expected else 0.0
    f1 = (
        2 * precision * recall / (precision + recall)
        if precision + recall
        else 0.0
    )

    return {
        "correct_tags": sorted(correct),
        "missing_tags": sorted(expected - generated),
        "extra_tags": sorted(generated - expected),
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "exact_match": expected == generated,
    }

## 6. Um caso, dois modelos

In [ ]:
print(MODELS)

In [ ]:
dataset_df.loc[0,"example_id"]

In [ ]:
prediction_model_1 = predict_one(dataset_df.loc[0,"example_id"], MODELS[0])
prediction_model_1

In [ ]:
prediction_model_2 = predict_one(dataset_df.loc[0,"example_id"], MODELS[1])
prediction_model_2

In [ ]:
single_case_rows = []
for prediction in [prediction_model_1, prediction_model_2]:
    metrics = compare_tags(
        prediction["expected_tags"],
        prediction["generated_tags"],
    )
    single_case_rows.append({**prediction, **metrics})

single_case_df = pd.DataFrame(single_case_rows)
single_case_df[[
    "model", "expected_tags", "generated_tags",
    "precision", "recall", "f1", "latency_ms",
]]

## 7. Rodar N exemplos por modelo

A amostra é a mesma para os dois modelos. As requisições são sequenciais, como definido para os benchmarks deste projeto.

In [ ]:
sample_ids = (
    dataset_df["example_id"]
    .sample(
        n=min(N_REQUESTS_PER_MODEL, len(dataset_df)),
        random_state=RANDOM_STATE,
    )
    .tolist()
)

batch_rows = []
for model in MODELS:
    for example_id in tqdm(sample_ids, desc=model):
        prediction = predict_one(example_id, model)
        metrics = compare_tags(
            prediction["expected_tags"],
            prediction["generated_tags"],
        )
        batch_rows.append({**prediction, **metrics})

predictions_df = pd.DataFrame(batch_rows)
predictions_df[[
    "example_id", "model", "expected_tags", "generated_tags",
    "precision", "recall", "f1", "exact_match", "latency_ms",
]]

## 8. Consolidar predições e dados originais

In [ ]:
metadata_columns = [
    "example_id", "domain", "split", "label_source",
    "taxonomy_version", "business_model_chars",
]

evaluated_df = predictions_df.merge(
    dataset_df[metadata_columns],
    on="example_id",
    how="left",
)

evaluated_df.head()

## 9. Macro F1 por modelo

O F1 individual mede cada exemplo. O macro F1 abaixo binariza as tags e dá o mesmo peso a cada tag observada no lote.

In [ ]:
def macro_f1_by_model(evaluated):
    label_space = sorted(
        {tag for tags in evaluated["expected_tags"] for tag in tags}
        | {tag for tags in evaluated["generated_tags"] for tag in tags}
    )
    binarizer = MultiLabelBinarizer(classes=label_space).fit([[]])

    rows = []
    for model, group in evaluated.groupby("model"):
        y_true = binarizer.transform(group["expected_tags"])
        y_pred = binarizer.transform(group["generated_tags"])
        rows.append({
            "model": model,
            "macro_f1": f1_score(
                y_true, y_pred, average="macro", zero_division=float("nan")
            ),
        })

    return pd.DataFrame(rows)

In [ ]:
performance_df = (
    evaluated_df.groupby("model", as_index=False)
    .agg(
        requests=("example_id", "size"),
        parse_rate=("parse_ok", "mean"),
        mean_individual_f1=("f1", "mean"),
        exact_match_rate=("exact_match", "mean"),
        mean_latency_ms=("latency_ms", "mean"),
    )
    .merge(macro_f1_by_model(evaluated_df), on="model")
    .sort_values("macro_f1", ascending=False)
)

performance_df